![bse_logo_textminingcourse](https://bse.eu/sites/default/files/bse_logo_small.png)

# Text Mining: Models and Algorithms
## NLP Analysis of Obligation Language in Preferential Trade Agreements — v4

**Project:** Legalization in International Trade — How Bilateral Relationship Features Predict Obligation Language Hardness
**Authors:** Apueela Wekulom, Jillian Hunter, Rhea D'Costa — Barcelona School of Economics

**Data source:** UNCTAD Texts of Trade Agreements (ToTA)
https://github.com/mappingtreaties/tota

### Research question
Do country pairs with stronger prior bilateral relationships — shared language, colonial history, and political alignment — use harder, more binding obligation language in the preferential trade agreements they sign?

### Notebook structure (v4 ordering)
1. Setup & imports
2. Corpus download (GitHub API → XML → parse)
3. Asymmetry flags & WTO card enrichment (immediately after download)
4. English-only filter & corpus description
5. Chapter name normalization (before scoring)
6. Preprocessing / stopwords (Figure 1)
7. Obligation scoring — improved dictionaries with longest-match-first & MIXED list
8. oblig_ratio distribution check
9. Agreement-level aggregation + oblig_ratio_core (Trade in Goods only)
10. TF-IDF bigram validation (Figure 2)
11. Chapter-level figures (Figure 3: by chapter type)
12. Obligation trend figure (Figure 4: over time)
13. Hofmann et al. external validation (Figure 5: scatter vs enforceable provision count)
14. SKELETON — Dyadic variable merge (CEPII + Bailey et al.)
15. SKELETON — Extension regression
16. Save outputs

### Key changes from v3
- Obligation scoring: longest-match-first with semantic priority (MIXED → SOFT → HARD)
- OBLIGATION_MIXED dict: "shall endeavour"-type phrases treated as soft
- "may" and "consider" removed from OBLIGATION_SOFT (permissive/procedural, not aspiration)
- "days" removed from ENFORCEMENT dict (too diffuse — fires on all deadline articles)
- Added hard obligation phrases from Hofmann et al. enforceability framework
- Chapter normalization now runs BEFORE scoring so all figures use normalized names
- oblig_ratio_core: restricted to Trade in Goods chapters only (NaN if absent)
- Hofmann et al. validation using wto_rta_id merge key
- Asymmetry flags run immediately after download
- Similarity network removed (not relevant to main paper)


---
## Part 0: Setup & Imports

In [ ]:
# Imports
import requests
import base64
import xml.etree.ElementTree as ET
import pandas as pd
import os
import re
import time
import numpy as np
from collections import Counter

# NLP
import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.util import ngrams as nltk_ngrams
nltk.download('punkt', quiet=True)

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['figure.dpi'] = 130

# ── GitHub token ───────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')

GITHUB_API = "https://api.github.com/repos/mappingtreaties/tota/contents/xml"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/109.0.0.0 Safari/537.36',
}
if GITHUB_TOKEN:
    HEADERS['Authorization'] = f'token {GITHUB_TOKEN}'
    print("\u2713 GitHub token loaded \u2014 authenticated (5,000 req/hr)")
else:
    print("\u26a0 No token found \u2014 unauthenticated (60 req/hr)")

_rl   = requests.get("https://api.github.com/rate_limit", headers=HEADERS).json()
_core = _rl.get('resources', {}).get('core', _rl.get('rate', {}))
print(f"  Rate limit: {_core.get('remaining','?')} / {_core.get('limit','?')} remaining")

OUTPUT_DIR = "rta_texts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nSetup complete. Output directory: {OUTPUT_DIR}")


---
## Part 1: Repository Structure

In [ ]:
# Step 1: fetch the directory listing from the GitHub API
response = requests.get(GITHUB_API, headers=HEADERS)
print("Response status:", response.status_code)
print("Rate limit remaining:", response.headers.get('X-RateLimit-Remaining', 'N/A'))

if response.status_code != 200:
    print(f"\nERROR response body: {response.text[:500]}")


In [ ]:
# Step 2: parse JSON
data = response.json()

if isinstance(data, dict):
    print("GitHub API returned an error:")
    print(f"  message: {data.get('message', 'no message')}")
    if 'rate limit' in str(data.get('message','')).lower():
        print("FIX: Set GITHUB_TOKEN in .env and restart kernel.")
    raise SystemExit("Fix error above then restart.")

file_list = data
print(f"Total files in xml/ folder: {len(file_list)}")


In [ ]:
# Step 3: extract file entries and sort numerically
xml_files = [f for f in file_list if f['name'].endswith('.xml')]
xml_files_sorted = sorted(
    xml_files,
    key=lambda f: int(re.search(r'pta_(\d+)', f['name']).group(1))
)
print(f"XML files found: {len(xml_files_sorted)}")
print(f"First: {xml_files_sorted[0]['name']}  |  Last: {xml_files_sorted[-1]['name']}")


---
## Part 2: Inspect One XML File

Inspect structure before looping — same class principle as examining `soup.prettify()` first.

In [ ]:
# Fetch and inspect pta_1.xml
test_file = xml_files_sorted[0]
r = requests.get(test_file['url'], headers=HEADERS)
file_json = r.json()
xml_bytes = base64.b64decode(file_json['content'])
xml_text  = xml_bytes.decode('utf-8')

root = ET.fromstring(xml_text)
meta = root.find('meta')
body = root.find('body')
chapters = body.findall('chapter')

print("Root tag:", root.tag)
print("\n=== META ===")
for child in meta:
    if child.tag == 'parties':
        parties = [p.text.strip() for p in child.findall('partyisocode') if p.text]
        print(f"  parties: {parties}")
    elif child.tag not in ('parties_original',):
        print(f"  {child.tag}: {child.text}")

print(f"\n=== BODY: {len(chapters)} chapters ===")
for ch in chapters:
    print(f"  Chapter: '{ch.get('name')}' \u2014 {len(ch.findall('article'))} articles")


---
## Part 3: Parse Function

In [ ]:
# ── Annex exclusion keywords ──────────────────────────────────────────────────
ANNEX_KEYWORDS = {'annex', 'schedule', 'appendix', 'attachment', 'exhibit'}

def parse_tota_xml(xml_text, pta_id, exclude_annexes=False):
    """
    Parses one ToTA XML string. Returns (meta_dict, articles_list).

    Parameters
    ----------
    xml_text        : str   raw XML content
    pta_id          : int   numeric ID from filename
    exclude_annexes : bool  if True, skip annex/schedule/appendix chapters.
                            Set False (default) to match Alschner et al. baseline.
    """
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError as e:
        print(f"  XML parse error in pta_{pta_id}: {e}")
        return None, []

    meta = root.find('meta')
    if meta is None:
        return None, []

    name          = meta.findtext('name',            default='').strip()
    rta_type      = meta.findtext('type',            default='').strip()
    status        = meta.findtext('status',          default='').strip()
    notification  = meta.findtext('notification',    default='').strip()
    date_signed   = meta.findtext('date_signed',     default='').strip()
    date_force    = meta.findtext('date_into_force', default='').strip()
    composition   = meta.findtext('composition',     default='').strip()
    region        = meta.findtext('region',          default='').strip()
    crossregional = meta.findtext('crossregional',   default='').strip()
    parties_wto   = meta.findtext('parties_wto',     default='').strip()
    language      = meta.findtext('language',        default='').strip()
    wto_rta_raw   = meta.findtext('wto_rta_id',      default='').strip()

    year = date_signed[:4] if date_signed else meta.findtext('year', default='').strip()

    try:
        wto_rta_id = int(wto_rta_raw) if wto_rta_raw else None
    except ValueError:
        wto_rta_id = None

    parties_el = meta.find('parties')
    parties = ([p.text.strip() for p in parties_el.findall('partyisocode')
                if p.text and p.text.strip()]
               if parties_el is not None else [])

    meta_dict = {
        'pta_id': pta_id, 'name': name, 'parties': parties,
        'n_parties': len(parties), 'year': year,
        'date_signed': date_signed, 'date_into_force': date_force,
        'status': status, 'type': rta_type, 'notification': notification,
        'composition': composition, 'region': region,
        'crossregional': crossregional, 'parties_wto': parties_wto,
        'language': language, 'wto_rta_id': wto_rta_id,
    }

    body = root.find('body')
    if body is None:
        return meta_dict, []

    articles = []
    n_annex_skipped = 0

    for ch_idx, chapter in enumerate(body.findall('chapter')):
        chapter_name = chapter.get('name', f'Chapter_{ch_idx}')

        if exclude_annexes:
            if any(kw in chapter_name.lower() for kw in ANNEX_KEYWORDS):
                n_annex_skipped += len(chapter.findall('article'))
                continue

        for article in chapter.findall('article'):
            art_text   = re.sub(r'\s+', ' ', ' '.join(article.itertext())).strip()
            word_count = len(art_text.split())
            if word_count > 0:
                articles.append({
                    'pta_id':         pta_id,
                    'agreement_name': name,
                    'year':           year,
                    'chapter_name':   chapter_name,
                    'chapter_index':  ch_idx,
                    'article_number': article.get('number', ''),
                    'article_name':   article.get('name',   ''),
                    'article_text':   art_text,
                    'word_count':     word_count,
                })

    return meta_dict, articles


# ── Quick self-test ────────────────────────────────────────────────────────────
_meta, _arts = parse_tota_xml(xml_text, pta_id=1)
print(f"pta_1 parsed: {len(_arts)} articles, parties={_meta['parties']}, wto_rta_id={_meta['wto_rta_id']}")


---
## Part 4: Corpus Download

In [ ]:
# Rate limit check
_rl = requests.get("https://api.github.com/rate_limit", headers=HEADERS).json()
_core = _rl.get('resources', {}).get('core', _rl.get('rate', {}))
remaining = _core.get('remaining', 0)
limit     = _core.get('limit', 0)
print(f"Rate limit: {remaining} / {limit} remaining")
if limit >= 5000:
    print(f"\u2713 Authenticated. Enough for full corpus ({len(xml_files_sorted)} files).")
else:
    print(f"\u26a0 Unauthenticated. Set GITHUB_TOKEN in .env for full corpus run.")


In [ ]:
META_COLS = [
    'pta_id', 'name', 'parties', 'n_parties', 'year',
    'date_signed', 'date_into_force', 'status', 'type',
    'notification', 'composition', 'region', 'crossregional',
    'parties_wto', 'language', 'wto_rta_id',
]
ARTICLE_COLS = [
    'pta_id', 'agreement_name', 'year', 'chapter_name', 'chapter_index',
    'article_number', 'article_name', 'article_text', 'word_count',
]

def download_tota_corpus(file_list, max_ptas=None, sleep=0.5,
                         save_xml=True, exclude_annexes=False):
    """Downloads and parses ToTA XML files from GitHub API."""
    all_meta, all_articles, errors = [], [], []
    targets = file_list[:max_ptas] if max_ptas else file_list
    total   = len(targets)
    print(f"Downloading {total} PTAs  |  exclude_annexes={exclude_annexes}")
    print("=" * 60)

    for i, f in enumerate(targets):
        fname  = f['name']
        pta_id = int(re.search(r'pta_(\d+)', fname).group(1))
        r = requests.get(f['url'], headers=HEADERS)

        if (i + 1) % 10 == 0 or i < 5:
            print(f"[{i+1:3}/{total}] pta_{pta_id:3}  status={r.status_code}  "                  f"rate={r.headers.get('X-RateLimit-Remaining','?')}")

        if r.status_code != 200:
            errors.append({'pta_id': pta_id, 'status': r.status_code})
            time.sleep(sleep); continue

        try:
            xml_bytes = base64.b64decode(r.json()['content'])
            xml_text  = xml_bytes.decode('utf-8')
        except Exception as e:
            errors.append({'pta_id': pta_id, 'status': 'decode_error'})
            time.sleep(sleep); continue

        if save_xml:
            with open(os.path.join(OUTPUT_DIR, fname), 'w', encoding='utf-8') as out:
                out.write(xml_text)

        meta_dict, articles = parse_tota_xml(xml_text, pta_id, exclude_annexes)
        if meta_dict:
            all_meta.append(meta_dict)
        all_articles.extend(articles)
        time.sleep(sleep)

    df_meta     = pd.DataFrame(all_meta,     columns=META_COLS) if all_meta     else pd.DataFrame(columns=META_COLS)
    df_articles = pd.DataFrame(all_articles, columns=ARTICLE_COLS) if all_articles else pd.DataFrame(columns=ARTICLE_COLS)
    print(f"\nDone. Agreements: {len(df_meta)} | Articles: {len(df_articles)} | Errors: {len(errors)}")
    return df_meta, df_articles, errors

print("download_tota_corpus() defined.")


In [ ]:
# ── PILOT RUN (100 agreements) ────────────────────────────────────────────────
# Change max_ptas=None for full corpus (requires GitHub token)
df_meta_pilot, df_articles_pilot, errors_pilot = download_tota_corpus(
    xml_files_sorted, max_ptas=100, sleep=0.8, save_xml=True
)
print("\nMetadata sample:")
display(df_meta_pilot[['pta_id', 'name', 'parties', 'year', 'wto_rta_id']].head(8))


In [ ]:
# ── FULL CORPUS RUN (uncomment when ready) ───────────────────────────────────
# df_meta, df_articles, errors = download_tota_corpus(
#     xml_files_sorted, max_ptas=None, sleep=0.5, save_xml=True
# )

# Use pilot data until full run is complete
df_meta     = df_meta_pilot.copy()
df_articles = df_articles_pilot.copy()
print(f"Working with: {len(df_meta)} agreements, {len(df_articles)} articles")


---
## Part 5: Power Asymmetry & Template Flags

Run immediately after download so flags are available throughout the notebook.
Grounds: Alschner et al. (2018) Figures 4 and 5 — post-TPA US agreements and
EU templates produce high obligation hardness by construction, not because of
the bilateral relationship. Template dummies are essential controls in the regression.


In [ ]:
import sys, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath('pta_asymmetry.py')))
import pta_asymmetry
importlib.reload(pta_asymmetry)
from pta_asymmetry import add_asymmetry_flags, asymmetry_summary, regression_controls

df_meta = add_asymmetry_flags(df_meta)
asymmetry_summary(df_meta)


---
## Part 6: WTO Card Metadata Enrichment

Scrape WTO card pages using wto_rta_id already in df_meta (from ToTA XML).
This enriches df_meta with agreement type, coverage, and status fields.
wto_rta_id is also the merge key for the Hofmann et al. validation in Part 13.


In [ ]:
from bs4 import BeautifulSoup
WTO_CARD_BASE = "https://rtais.wto.org/UI/PublicShowRTAIDCard.aspx?rtaid="

def scrape_wto_card(rta_id, session=None):
    """Scrape one WTO RTA card page (plain GET, no ViewState needed)."""
    url = WTO_CARD_BASE + str(rta_id)
    try:
        req = session if session else requests
        r   = req.get(url, timeout=20)
        r.raise_for_status()
    except Exception as e:
        return None
    soup   = BeautifulSoup(r.text, 'html.parser')
    record = {'wto_rta_id': rta_id}
    for row in soup.find_all('tr'):
        cells = row.find_all(['td', 'th'])
        if len(cells) == 2:
            key = re.sub(r'[^a-z0-9]+', '_', cells[0].get_text(strip=True).lower()).strip('_')
            val = cells[1].get_text(separator=' ', strip=True)
            if key:
                record[key] = val
    return record

# Scrape WTO cards for all agreements with a valid wto_rta_id
wto_session = requests.Session()
wto_session.headers.update(HEADERS)
wto_records, wto_errors = [], []
rta_ids = df_meta['wto_rta_id'].dropna().astype(int).tolist()
print(f"Fetching {len(rta_ids)} WTO card pages...")

for i, rta_id in enumerate(rta_ids):
    record = scrape_wto_card(rta_id, wto_session)
    if record:
        wto_records.append(record)
    else:
        wto_errors.append(rta_id)
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/{len(rta_ids)}] collected={len(wto_records)}")
    time.sleep(0.4)

df_wto = pd.DataFrame(wto_records)
df_meta = df_meta.merge(df_wto, on='wto_rta_id', how='left')
print(f"\nWTO enrichment done. {len(wto_records)} cards scraped, {len(wto_errors)} errors.")
print(f"df_meta columns: {list(df_meta.columns)}")


---
## Part 7: English Filter & Corpus Description

In [ ]:
# ── English-only filter ───────────────────────────────────────────────────────
if 'language' not in df_articles.columns:
    df_articles = df_articles.merge(df_meta[['pta_id', 'language']], on='pta_id', how='left')
df_articles['language'] = df_articles['language'].fillna('en')

n_before = len(df_articles)
en_ptas  = set(df_meta[df_meta['language'] == 'en']['pta_id'])
df_articles = df_articles[df_articles['pta_id'].isin(en_ptas)].reset_index(drop=True)
n_dropped_non_english = n_before - len(df_articles)

print(f"English-only filter: kept {len(en_ptas)} agreements, {len(df_articles)} articles")
print(f"Dropped {n_dropped_non_english} articles from non-English PTAs")

# ── Corpus summary ─────────────────────────────────────────────────────────────
df_agree_summary = df_articles.groupby(['pta_id', 'agreement_name', 'year']).agg(
    n_articles  = ('article_text', 'count'),
    n_chapters  = ('chapter_name', 'nunique'),
    total_words = ('word_count', 'sum'),
).reset_index()
df_agree_summary['year'] = pd.to_numeric(df_agree_summary['year'], errors='coerce')

print(f"\n=== CORPUS SUMMARY ===")
print(f"Agreements : {len(df_agree_summary)}")
print(f"Articles   : {len(df_articles)}")
print(f"Total words: {df_agree_summary['total_words'].sum():,}")


---
## Part 8: Chapter Name Normalization

**Runs before scoring** so all downstream analysis — figures, aggregation,
oblig_ratio_core — uses harmonized chapter categories.

Based on Alschner et al. (2018) 73-category taxonomy, collapsed to ~18
categories that appear frequently enough for cross-agreement comparison.


In [ ]:
CHAPTER_TAXONOMY = [
    # Dispute settlement — highest priority (match before 'trade')
    (['dispute', 'settlement of dispute', 'arbitrat', 'controversia',
      'differend', 'panel', 'resolut'],                        'Dispute Settlement'),
    # Investment
    (['investment', 'investor', 'capital movement',
      'right of establishment'],                               'Investment'),
    # Intellectual property
    (['intellectual property', 'ip ', 'copyright', 'patent',
      'trademark', 'trips'],                                   'Intellectual Property'),
    # Competition & subsidies
    (['competition', 'anti-trust', 'antitrust', 'monopol',
      'state aid', 'subsid'],                                  'Competition & Subsidies'),
    # Government procurement
    (['government procurement', 'public procurement'],         'Government Procurement'),
    # Trade in services
    (['service', 'gats', 'financial service',
      'telecommunication', 'e-commerce', 'digital'],           'Trade in Services'),
    # Rules of origin
    (['rules of origin', 'origin', 'cumulation'],              'Rules of Origin'),
    # TBT & SPS
    (['technical barrier', 'tbt', 'sanitary', 'phytosanitary',
      'sps', 'standard'],                                      'TBT & SPS'),
    # Customs & trade facilitation
    (['custom', 'tariff', 'duty', 'import', 'export',
      'trade facilitat'],                                      'Customs & Trade Facilitation'),
    # Trade in goods — after customs so 'goods' doesn't match customs
    (['trade in goods', 'goods', 'market access',
      'industrial product'],                                   'Trade in Goods'),
    # Labour
    (['labour', 'labor', 'worker', 'employment', 'social'],    'Labour'),
    # Environment
    (['environment', 'sustainable', 'climate'],                'Environment'),
    # Transparency
    (['transparency', 'publication', 'information'],           'Transparency'),
    # Institutional
    (['institution', 'committee', 'joint', 'administration',
      'management'],                                           'Institutional Framework'),
    # Exceptions & safeguards
    (['exception', 'general provision', 'carve', 'safeguard'], 'Exceptions & Safeguards'),
    # Preamble & objectives
    (['preamble', 'objective', 'purpose', 'goal',
      'recital', 'introduct'],                                 'Preamble & Objectives'),
    # Final / administrative
    (['final provision', 'administrative', 'miscellaneous',
      'general and final'],                                    'Administrative & Final'),
    # Cooperation / development
    (['cooperation', 'technical assist', 'capacity',
      'development'],                                          'Cooperation & Development'),
]

def normalize_chapter(raw_name):
    """Map raw chapter name to standardized taxonomy label."""
    name_lower = str(raw_name).lower().strip()
    for keywords, label in CHAPTER_TAXONOMY:
        if any(kw in name_lower for kw in keywords):
            return label
    return f'Other \u2014 {raw_name}'

df_articles['chapter_norm'] = df_articles['chapter_name'].apply(normalize_chapter)

# Coverage check
n_other = df_articles['chapter_norm'].str.startswith('Other').sum()
print(f"Chapter normalization complete.")
print(f"  Classified  : {len(df_articles) - n_other} articles ({100*(len(df_articles)-n_other)/len(df_articles):.1f}%)")
print(f"  Other/unknown: {n_other} articles")
print(f"\nChapter distribution (top 15):")
print(df_articles['chapter_norm'].value_counts().head(15).to_string())


---
## Part 9: Preprocessing & Figure 1 (Stopword Effect)

In [ ]:
# ── Import domain stopwords (trade_stopwords.py v4) ───────────────────────────
import sys, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath('trade_stopwords.py')))
import trade_stopwords
importlib.reload(trade_stopwords)
from trade_stopwords import get_stopwords, LAYER_3_PROTECT, describe_layers

TRADE_STOPWORDS = get_stopwords()
describe_layers()

# ── Helper functions ───────────────────────────────────────────────────────────
def strip_punct(word):
    return re.sub(r'\W+', '', word)

def abbr_or_lower(word):
    """Keep abbreviations uppercase (WTO, FTA), lowercase everything else."""
    if re.match(r'([A-Z]+[a-z]*){2,}', word):
        return word
    return word.lower()

def preprocess(text, remove_stops=True, stem=False):
    """Tokenise and clean one article string."""
    words  = text.split()
    tokens = [abbr_or_lower(strip_punct(w)) for w in words]
    tokens = [t for t in tokens if len(t) > 1]
    if remove_stops:
        tokens = [t for t in tokens if t.lower() not in TRADE_STOPWORDS]
    if stem:
        stemmer = SnowballStemmer('english')
        tokens  = [stemmer.stem(t) for t in tokens]
    return tokens

df_articles['tokens'] = df_articles['article_text'].apply(preprocess)
print(f"Tokenization complete. Sample token count: {df_articles['tokens'].apply(len).mean():.1f} tokens/article")


In [ ]:
# ── Figure 1: Preprocessing before/after ─────────────────────────────────────
all_tokens_raw   = [t for tokens in df_articles['article_text'].apply(
    lambda t: preprocess(t, remove_stops=False)) for t in tokens]
all_tokens_clean = [t for tokens in df_articles['tokens'] for t in tokens]

freq_raw   = Counter(all_tokens_raw).most_common(20)
freq_clean = Counter(all_tokens_clean).most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 1: Effect of Domain Stopword Removal on Term Frequency\n'
             '(obligation terms survive; boilerplate removed)',
             fontsize=11, fontweight='bold')

for ax, (freq, title, col) in zip(axes, [
    (freq_raw,   'Before: raw frequencies',                    '#D0D7E3'),
    (freq_clean, 'After: domain stopwords removed (v4)',       '#2E5FA3')
]):
    words, counts = zip(*freq)
    ax.barh(list(words)[::-1], list(counts)[::-1], color=col, alpha=0.9)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Frequency')
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_preprocessing.png'), bbox_inches='tight')
plt.show()
print("Figure 1 saved.")


---
## Part 10: Obligation Language Scoring (v4)

### Key improvements over v3
**Longest-match-first with semantic priority:**
Matching order: OBLIGATION_MIXED → OBLIGATION_SOFT → OBLIGATION_HARD
This prevents double-counting and correctly handles mixed phrases.

**OBLIGATION_MIXED:** "shall endeavour"-type phrases where a hard grammatical
vehicle carries soft substantive content. Grounded in international legal
drafting conventions — these are best-efforts obligations, not binding ones.
Counted as soft in the final ratio.

**Removed from OBLIGATION_SOFT:** "may" (permissive, not aspirational per
Vienna Convention) and "consider" (procedural instruction).

**Removed from ENFORCEMENT:** "days" (fires on all deadline articles —
too diffuse to carry enforcement signal).

**Added to OBLIGATION_HARD:** phrases from Hofmann et al. enforceability
framework: "shall be entitled to", "shall have the right to", "shall be
subject to".


In [ ]:
# ── Obligation dictionaries (v4) ──────────────────────────────────────────────

# MIXED: hard grammatical vehicle + soft substantive content
# Matched FIRST — consumed before SOFT or HARD matching runs.
# Grounding: Vienna Convention drafting guidance; Abbott et al. (2000)
# treat these as nominally binding but substantively aspirational.
OBLIGATION_MIXED = [
    'shall endeavour', 'shall endeavor',
    'shall seek to', 'shall promote',
    'shall encourage', 'shall aim to',
    'shall strive', 'shall facilitate',
    'shall cooperate', 'shall take into account',
    'shall have regard to', 'shall consider',
    'shall work towards', 'shall work toward',
    'shall use best efforts', 'shall make every effort',
]

# SOFT: pure aspiration / best-efforts markers
# "may" and "consider" REMOVED (permissive/procedural — see stopwords docstring)
OBLIGATION_SOFT = [
    'should', 'endeavour', 'endeavor',
    'endeavours', 'endeavors',
    'encourage', 'promotes', 'promoted',
    'best efforts', 'best endeavours',
    'where possible', 'where feasible',
    'to the extent possible', 'to the extent practicable',
    'seek to', 'seeks to',
    'strive', 'strives',
    'aim to', 'aims to',
]

# HARD: binding obligation markers
# Added: "shall be entitled to", "shall have the right to", "shall be subject to"
# (Hofmann et al. enforceability framework grounding)
OBLIGATION_HARD = [
    'shall', 'must',
    'is required', 'are required', 'required to',
    'obliged to', 'obligated to',
    'shall ensure', 'must ensure', 'will ensure',
    'shall be prohibited', 'is prohibited', 'are prohibited',
    'shall not', 'must not',
    'undertakes', 'undertakes to', 'undertake to',
    'commits to', 'commit to', 'commits',
    'binds', 'bound to', 'bound by',
    'shall be entitled to', 'shall have the right to',
    'shall be subject to',
]

# ENFORCEMENT: DS machinery and consequence language
# "days" REMOVED — fires on all deadline articles, too diffuse
ENFORCEMENT = [
    'dispute', 'arbitration', 'arbitral', 'arbitrator',
    'panel', 'tribunal',
    'remedy', 'remedies',
    'sanction', 'sanctions',
    'compensation', 'compensate',
    'retaliation', 'retaliatory',
    'suspension', 'suspended',
    'countermeasure', 'countermeasures',
    'enforce', 'enforcement', 'enforceable',
    'comply', 'compliance', 'non-compliance',
    'violation', 'violate', 'breach',
    'ruling', 'rulings', 'finding', 'findings',
    'appellate', 'appeal',
    'consultations',
    'binding ruling', 'final ruling',
]

# ── Sanity check ───────────────────────────────────────────────────────────────
core_check = ['shall', 'must', 'endeavour', 'arbitration', 'compliance', 'dispute']
missing = [t for t in core_check if t not in LAYER_3_PROTECT]
if missing:
    print(f'\u26a0 Warning: these core terms not in LAYER_3_PROTECT: {missing}')
else:
    print('\u2713 Core dictionary terms confirmed in LAYER_3_PROTECT')

# Check MIXED terms not in HARD or SOFT (they shouldn't be)
mixed_in_hard = [t for t in OBLIGATION_MIXED if t in OBLIGATION_HARD]
mixed_in_soft = [t for t in OBLIGATION_MIXED if t in OBLIGATION_SOFT]
if mixed_in_hard or mixed_in_soft:
    print(f'\u26a0 MIXED terms found in HARD: {mixed_in_hard}')
    print(f'\u26a0 MIXED terms found in SOFT: {mixed_in_soft}')
else:
    print('\u2713 OBLIGATION_MIXED terms do not overlap with HARD or SOFT')

print(f'\nDictionary sizes: MIXED={len(OBLIGATION_MIXED)}, SOFT={len(OBLIGATION_SOFT)}, HARD={len(OBLIGATION_HARD)}, ENF={len(ENFORCEMENT)}')


In [ ]:
# ── dict_score_v4: longest-match-first with semantic priority ─────────────────
#
# Algorithm:
# 1. Work on lowercased text copy
# 2. Sort each list longest-first (greedy matching)
# 3. Match MIXED first — replace matched spans with placeholder to prevent
#    re-matching by SOFT or HARD
# 4. Match SOFT on remaining text
# 5. Match HARD on remaining text
# 6. ENFORCEMENT scored independently on original text (no interaction)
#
# Placeholders use unlikely character sequence to avoid false matches.

PLACEHOLDER = '\x00MATCHED\x00'

def _consume(text, term_list):
    """
    Match terms in text (longest first), count hits, replace with placeholder.
    Returns (count, text_with_placeholders).
    """
    terms_sorted = sorted(term_list, key=len, reverse=True)
    count = 0
    for term in terms_sorted:
        hits = text.count(term)
        if hits > 0:
            count += hits
            text = text.replace(term, PLACEHOLDER)
    return count, text

def dict_score_v4(text, mixed_list, soft_list, hard_list, enf_list):
    """
    Score one article with longest-match-first semantic priority.

    Priority: MIXED -> SOFT -> HARD (on progressively consumed text)
    ENFORCEMENT scored independently.

    Returns dict with raw counts and per-1000-word rates.
    """
    n_words = len(text.split())
    if n_words == 0:
        return {
            'mixed_raw': 0, 'soft_raw': 0, 'hard_raw': 0, 'enf_raw': 0,
            'mixed_p1k': 0, 'soft_p1k': 0, 'hard_p1k': 0, 'enf_p1k': 0,
        }

    text_lower = text.lower()

    # Step 1: match MIXED (soft compounds) — consume first
    mixed_raw, text_remaining = _consume(text_lower, mixed_list)

    # Step 2: match SOFT on remaining text
    soft_raw, text_remaining = _consume(text_remaining, soft_list)

    # Step 3: match HARD on remaining text
    hard_raw, _ = _consume(text_remaining, hard_list)

    # Step 4: enforcement scored on original text (independent)
    enf_raw, _ = _consume(text_lower, enf_list)

    def p1k(raw):
        return (raw / n_words * 1000) if n_words > 0 else 0

    return {
        'mixed_raw': mixed_raw, 'soft_raw':  soft_raw,
        'hard_raw':  hard_raw,  'enf_raw':   enf_raw,
        'mixed_p1k': p1k(mixed_raw), 'soft_p1k': p1k(soft_raw),
        'hard_p1k':  p1k(hard_raw),  'enf_p1k':  p1k(enf_raw),
    }

# ── Apply scoring ──────────────────────────────────────────────────────────────
print("Scoring articles...")
scores = df_articles['article_text'].apply(
    lambda t: pd.Series(dict_score_v4(t, OBLIGATION_MIXED, OBLIGATION_SOFT,
                                       OBLIGATION_HARD, ENFORCEMENT))
)
df_articles = pd.concat([df_articles, scores], axis=1)

# oblig_ratio: hard / (soft_total + 1)
# soft_total = soft_raw + mixed_raw (mixed counted as soft in ratio)
df_articles['soft_total'] = df_articles['soft_raw'] + df_articles['mixed_raw']
df_articles['oblig_ratio'] = df_articles['hard_raw'] / (df_articles['soft_total'] + 1)

print(f"Scoring complete.")
print(f"  hard_p1k mean : {df_articles['hard_p1k'].mean():.2f}")
print(f"  soft_p1k mean : {df_articles['soft_p1k'].mean():.2f}")
print(f"  mixed_p1k mean: {df_articles['mixed_p1k'].mean():.2f}")
print(f"  enf_p1k mean  : {df_articles['enf_p1k'].mean():.2f}")
print(f"  oblig_ratio mean: {df_articles['oblig_ratio'].mean():.3f}")


---
## Part 11: oblig_ratio Distribution Check

Validation step before producing any interpretive figures.

In [ ]:
# ── Agreement-level aggregation ───────────────────────────────────────────────
def w_mean(g, val, wt):
    """Length-weighted mean."""
    return (g[val] * g[wt]).sum() / g[wt].sum() if g[wt].sum() > 0 else 0

df_agreement = df_articles.groupby(['pta_id', 'agreement_name', 'year']).apply(
    lambda g: pd.Series({
        'total_words':    g['word_count'].sum(),
        'n_articles':     len(g),
        'n_chapters':     g['chapter_norm'].nunique(),
        'hard_p1k_mean':  w_mean(g, 'hard_p1k',   'word_count'),
        'soft_p1k_mean':  w_mean(g, 'soft_p1k',   'word_count'),
        'mixed_p1k_mean': w_mean(g, 'mixed_p1k',  'word_count'),
        'enf_p1k_mean':   w_mean(g, 'enf_p1k',    'word_count'),
        'oblig_ratio':    w_mean(g, 'oblig_ratio', 'word_count'),
        'hard_total':     g['hard_raw'].sum(),
        'soft_total':     g['soft_total'].sum(),
    })
).reset_index()
df_agreement['year'] = pd.to_numeric(df_agreement['year'], errors='coerce')

# ── oblig_ratio_core: Trade in Goods chapters only ────────────────────────────
# NaN if agreement has no Trade in Goods chapter
tig_articles = df_articles[df_articles['chapter_norm'] == 'Trade in Goods']
tig_agg = tig_articles.groupby('pta_id').apply(
    lambda g: pd.Series({
        'oblig_ratio_core': w_mean(g, 'oblig_ratio', 'word_count'),
        'n_tig_articles': len(g),
    })
).reset_index()

df_agreement = df_agreement.merge(tig_agg, on='pta_id', how='left')
# NaN for agreements without Trade in Goods chapters (already handled by left join)
n_core_nan = df_agreement['oblig_ratio_core'].isna().sum()
print(f"oblig_ratio_core: {df_agreement['oblig_ratio_core'].notna().sum()} agreements have Trade in Goods chapters")
print(f"  NaN (no TiG chapter): {n_core_nan} agreements")

# ── Add asymmetry flags ────────────────────────────────────────────────────────
df_agreement = regression_controls(df_agreement, df_meta)

print(f"\nAgreement-level table: {len(df_agreement)} rows x {len(df_agreement.columns)} cols")
print("\noblig_ratio distribution:")
print(df_agreement['oblig_ratio'].describe().round(3).to_string())


In [ ]:
# ── Distribution sanity checks ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('oblig_ratio Distribution Check', fontsize=11, fontweight='bold')

# Histogram
ax = axes[0]
ax.hist(df_agreement['oblig_ratio'].dropna(), bins=30, color='#2E5FA3', alpha=0.85, edgecolor='white')
ax.axvline(df_agreement['oblig_ratio'].median(), color='red', linestyle='--', label='Median')
ax.set_xlabel('oblig_ratio (hard / soft_total + 1)')
ax.set_ylabel('Count')
ax.set_title('Full measure distribution')
ax.legend()

# Core vs full scatter
ax = axes[1]
merged_check = df_agreement.dropna(subset=['oblig_ratio', 'oblig_ratio_core'])
ax.scatter(merged_check['oblig_ratio'], merged_check['oblig_ratio_core'],
           alpha=0.6, s=20, color='#1F3864')
lim = max(merged_check['oblig_ratio'].max(), merged_check['oblig_ratio_core'].max()) + 0.1
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, label='1:1 line')
r_val = merged_check[['oblig_ratio', 'oblig_ratio_core']].corr().iloc[0, 1]
ax.set_xlabel('oblig_ratio (full)')
ax.set_ylabel('oblig_ratio_core (Trade in Goods only)')
ax.set_title(f'Full vs core measure (r={r_val:.3f}, n={len(merged_check)})')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_distribution_check.png'), bbox_inches='tight')
plt.show()

# ── Validation assertion ───────────────────────────────────────────────────────
ds_chapters = df_articles[df_articles['chapter_norm'] == 'Dispute Settlement']
coop_chapters = df_articles[df_articles['chapter_norm'] == 'Cooperation & Development']
if len(ds_chapters) > 0 and len(coop_chapters) > 0:
    ds_ratio   = ds_chapters['oblig_ratio'].mean()
    coop_ratio = coop_chapters['oblig_ratio'].mean()
    flag = '\u2713' if ds_ratio > coop_ratio else '\u26a0'
    print(f"\n{flag} DS chapter mean oblig_ratio: {ds_ratio:.3f}")
    print(f"  Cooperation chapter mean   : {coop_ratio:.3f}")
    print(f"  Expected: DS > Cooperation. {'PASS' if ds_ratio > coop_ratio else 'FAIL — check normalization'}")


---
## Part 12: TF-IDF Bigram Validation (Figure 2)

Data-driven validation: do agreements classified as high-ratio by the dictionary also use different vocabulary according to a fully independent method?

In [ ]:
# ── Phrase-aware preprocessing for n-gram analysis ───────────────────────────
PROTECTED_PHRASES = sorted(
    [t for t in LAYER_3_PROTECT if ' ' in t], key=len, reverse=True
)

def protect_phrases(text):
    t = text.lower()
    for phrase in PROTECTED_PHRASES:
        t = t.replace(phrase, phrase.replace(' ', '_'))
    return t

def preprocess_ngram(text):
    protected = protect_phrases(text)
    tokens = protected.split()
    tokens = [re.sub(r'\W+', '', t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    tokens = [t for t in tokens if t not in TRADE_STOPWORDS]
    return tokens

df_articles['tokens_phrase'] = df_articles['article_text'].apply(preprocess_ngram)

def get_ngrams(tokens, n):
    return ['_'.join(g) for g in nltk_ngrams(tokens, n)]

df_articles['bigrams'] = df_articles['tokens_phrase'].apply(lambda t: get_ngrams(t, 2))
print("Phrase-aware bigrams computed.")


In [ ]:
# ── TF-IDF: top vs bottom oblig_ratio quartile ────────────────────────────────
pta_bigram_docs = (
    df_articles.groupby('pta_id')['bigrams']
    .apply(lambda lists: ' '.join(bg for lst in lists for bg in lst))
    .reset_index()
    .rename(columns={'bigrams': 'bigram_doc'})
)
pta_bigram_docs = pta_bigram_docs.merge(
    df_agreement[['pta_id', 'agreement_name', 'oblig_ratio']], on='pta_id'
)

vec = TfidfVectorizer(max_features=600, min_df=3)
tfidf_matrix = vec.fit_transform(pta_bigram_docs['bigram_doc'])
feat = vec.get_feature_names_out()

q75 = pta_bigram_docs['oblig_ratio'].quantile(0.75)
q25 = pta_bigram_docs['oblig_ratio'].quantile(0.25)
mask_top = (pta_bigram_docs['oblig_ratio'] >= q75).values
mask_bot = (pta_bigram_docs['oblig_ratio'] <= q25).values

mean_top = tfidf_matrix[mask_top].mean(axis=0).A1
mean_bot = tfidf_matrix[mask_bot].mean(axis=0).A1

diff_deep     = mean_top - mean_bot
top20_deep    = diff_deep.argsort()[::-1][:20]
deep_labels   = [feat[i].replace('_', ' ') for i in top20_deep]
deep_scores   = diff_deep[top20_deep]

diff_shallow   = mean_bot - mean_top
top20_shallow  = diff_shallow.argsort()[::-1][:20]
shallow_labels = [feat[i].replace('_', ' ') for i in top20_shallow]
shallow_scores = diff_shallow[top20_shallow]

print(f"High-ratio bigrams (top quartile \u2265 {q75:.2f}, n={mask_top.sum()}):  {deep_labels[:5]}")
print(f"Low-ratio bigrams  (bot quartile \u2264 {q25:.2f}, n={mask_bot.sum()}):  {shallow_labels[:5]}")


In [ ]:
# ── Figure 2: Discriminating bigrams ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
fig.suptitle(
    'Figure 2: TF-IDF Validation — Discriminating Bigrams by Obligation Ratio Quartile\n'
    '(Independent data-driven check: does the dictionary measure agree with corpus vocabulary?)',
    fontsize=11, fontweight='bold'
)

ax1.barh(deep_labels[::-1], deep_scores[::-1], color='#1F3864', alpha=0.85)
ax1.set_xlabel('TF-IDF enrichment in high-ratio agreements')
ax1.set_title(f'Bigrams in HIGH-RATIO agreements\n(oblig_ratio \u2265 {q75:.2f}, n={mask_top.sum()})')
ax1.tick_params(axis='y', labelsize=9)
ax1.axvline(0, color='black', linewidth=0.8)

ax2.barh(shallow_labels[::-1], shallow_scores[::-1], color='#C0392B', alpha=0.75)
ax2.set_xlabel('TF-IDF enrichment in low-ratio agreements')
ax2.set_title(f'Bigrams in LOW-RATIO agreements\n(oblig_ratio \u2264 {q25:.2f}, n={mask_bot.sum()})')
ax2.tick_params(axis='y', labelsize=9)
ax2.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_tfidf_validation.png'), bbox_inches='tight')
plt.show()
print("Figure 2 saved.")


---
## Part 13: Figure 3 — Obligation Hardness by Chapter Type

Face validity check. Expected ordering: Dispute Settlement > Trade in Goods > Cooperation > Preamble.

In [ ]:
chapter_stats = df_articles.groupby('chapter_norm').agg(
    n_articles      = ('article_text', 'count'),
    avg_hard_p1k    = ('hard_p1k',    'mean'),
    avg_soft_p1k    = ('soft_p1k',    'mean'),
    avg_mixed_p1k   = ('mixed_p1k',   'mean'),
    avg_enf_p1k     = ('enf_p1k',     'mean'),
    avg_oblig_ratio = ('oblig_ratio', 'mean'),
    total_words     = ('word_count',  'sum'),
).reset_index()

# Keep chapters with at least 3 articles; exclude 'Other' catch-all
chapter_stats = chapter_stats[
    (chapter_stats['n_articles'] >= 3) &
    (~chapter_stats['chapter_norm'].str.startswith('Other'))
].sort_values('avg_oblig_ratio', ascending=True)

fig, ax = plt.subplots(figsize=(11, max(6, len(chapter_stats) * 0.4)))

short_ch = [str(c)[:40] for c in chapter_stats['chapter_norm']]
width = 0.25
x     = range(len(chapter_stats))

ax.barh([i - width for i in x], chapter_stats['avg_hard_p1k'],
        height=width, label='Hard (shall/must)', color='#1F3864', alpha=0.85)
ax.barh([i         for i in x], chapter_stats['avg_soft_p1k'] + chapter_stats['avg_mixed_p1k'],
        height=width, label='Soft + Mixed (should/shall endeavour)', color='#D0D7E3', alpha=0.85)
ax.barh([i + width for i in x], chapter_stats['avg_enf_p1k'],
        height=width, label='Enforcement (dispute/arbitration)', color='#2E5FA3', alpha=0.6)

ax.set_yticks(list(x))
ax.set_yticklabels(short_ch, fontsize=8)
ax.set_xlabel('Average frequency per 1,000 words')
ax.set_title('Figure 3: Obligation Language by Chapter Type\n'
             '(sorted by avg oblig_ratio — face validity check)', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_chapter_obligation.png'), bbox_inches='tight')
plt.show()
print("Figure 3 saved.")

# ── Face validity check ────────────────────────────────────────────────────────
ds_row   = chapter_stats[chapter_stats['chapter_norm'] == 'Dispute Settlement']
coop_row = chapter_stats[chapter_stats['chapter_norm'] == 'Cooperation & Development']
if len(ds_row) > 0 and len(coop_row) > 0:
    flag = '\u2713' if ds_row['avg_oblig_ratio'].values[0] > coop_row['avg_oblig_ratio'].values[0] else '\u26a0'
    print(f"\n{flag} Face validity: DS ratio={ds_row['avg_oblig_ratio'].values[0]:.3f}  Coop ratio={coop_row['avg_oblig_ratio'].values[0]:.3f}")


---
## Part 14: Figure 4 — Obligation Hardness Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Figure 4: Obligation Language Hardness in PTAs (1948\u20132015)\n'
             'Coloured by template flag', fontsize=11, fontweight='bold')

df_time = df_agreement.dropna(subset=['year', 'oblig_ratio']).copy()

template_colors = {
    'none':               '#1F3864',
    'us_tpa_template':    '#D85A30',
    'eu_template':        '#2E5FA3',
    'us_pre_tpa':         '#8FA3BF',
    'japan_epa_template': '#5DCAA5',
    'china_template':     '#EF9F27',
}
for flag, color in template_colors.items():
    mask = df_time['template_flag'] == flag
    if mask.sum() > 0:
        ax.scatter(df_time.loc[mask, 'year'], df_time.loc[mask, 'oblig_ratio'],
                   color=color, alpha=0.65, s=30, zorder=2,
                   label=f"{flag.replace('_',' ')} (n={mask.sum()})")

if len(df_time) >= 5:
    trend = (df_time.sort_values('year').set_index('year')['oblig_ratio']
             .rolling(window=5, min_periods=2).mean())
    ax.plot(trend.index, trend.values, color='black', linewidth=1.5,
            linestyle='--', label='5-yr rolling mean', zorder=3)

ax.set_xlabel('Year of signature')
ax.set_ylabel('Obligation ratio (hard / soft_total + 1)')
ax.set_title('Obligation hardness by template type')
ax.legend(fontsize=7, loc='upper left')
ax.axhline(df_time['oblig_ratio'].median(), color='grey', linestyle=':', linewidth=1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig4_hardness_over_time.png'), bbox_inches='tight')
plt.show()
print("Figure 4 saved.")


---
## Part 15: Figure 5 — Hofmann et al. External Validation

Merge Hofmann et al. (2017) Horizontal Depth database on wto_rta_id.
Hofmann et al. code each of 52 policy areas as legally enforceable or not
across 279 PTAs. We correlate our continuous oblig_ratio against their
enforceable provision count.

**Conceptual note:** Hofmann et al. measure whether provisions are *present
and formally enforceable* (binary, per policy area). Our measure captures
*how binding the language is* within whatever provisions exist (continuous).
These are related but distinct constructs — a moderate correlation confirms
overlap without redundancy. A very high correlation (r > 0.8) would suggest
redundancy; near zero would suggest construct invalidity.

**Merge key:** wto_rta_id (already in df_meta from ToTA XML, present in
Hofmann et al. dataset).


In [ ]:
# ── Load Hofmann et al. data ──────────────────────────────────────────────────
# Expected file: hofmann_horizontal_depth.csv (or .dta)
# Source: World Bank — http://data.worldbank.org/data-catalog/deep-trade-agreements
# Required columns: wto_rta_id (or equivalent ID), enf_count (enforceable provision count)
# If column names differ, rename below.

HOFMANN_PATH = 'hofmann_horizontal_depth.csv'   # update path if needed

if os.path.exists(HOFMANN_PATH):
    df_hofmann = pd.read_csv(HOFMANN_PATH)
    # Rename columns to standard names if needed
    # df_hofmann = df_hofmann.rename(columns={'rtaid': 'wto_rta_id', 'total_enforceable': 'enf_count'})
    print(f"Hofmann data loaded: {len(df_hofmann)} rows")
    print(f"Columns: {list(df_hofmann.columns)}")
else:
    print(f"\u26a0 Hofmann data not found at '{HOFMANN_PATH}'.")
    print("  Place hofmann_horizontal_depth.csv in the working directory.")
    print("  Download from: http://data.worldbank.org/data-catalog/deep-trade-agreements")
    df_hofmann = None


In [ ]:
# ── Merge and validate ────────────────────────────────────────────────────────
if df_hofmann is not None:
    # Merge on wto_rta_id
    df_val = df_agreement.merge(
        df_hofmann[['wto_rta_id', 'enf_count']],
        left_on='pta_id',   # placeholder — use actual key
        right_on='wto_rta_id',
        how='inner'
    )
    # More likely merge via df_meta wto_rta_id:
    df_val = df_agreement.merge(
        df_meta[['pta_id', 'wto_rta_id']], on='pta_id', how='left'
    ).merge(
        df_hofmann[['wto_rta_id', 'enf_count']], on='wto_rta_id', how='inner'
    )
    df_val = df_val.dropna(subset=['oblig_ratio', 'enf_count'])

    r, p = pearsonr(df_val['oblig_ratio'], df_val['enf_count'])
    print(f"Hofmann validation: n={len(df_val)} agreements")
    print(f"  Pearson r = {r:.3f}  (p = {p:.4f})")
    print(f"  Interpretation: {'moderate convergence \u2713' if 0.25 < abs(r) < 0.75 else 'check construct alignment'}")

    # ── Figure 5: scatter plot ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))

    # Colour by template
    colors = df_val['template_flag'].map({
        'none': '#1F3864', 'us_tpa_template': '#D85A30',
        'eu_template': '#2E5FA3'
    }).fillna('#8FA3BF')

    ax.scatter(df_val['enf_count'], df_val['oblig_ratio'],
               c=colors, alpha=0.7, s=40, edgecolors='white', linewidth=0.4)

    # Regression line
    m, b = np.polyfit(df_val['enf_count'], df_val['oblig_ratio'], 1)
    x_line = np.linspace(df_val['enf_count'].min(), df_val['enf_count'].max(), 100)
    ax.plot(x_line, m * x_line + b, 'k-', linewidth=1.5, alpha=0.6,
            label=f'OLS fit  r={r:.3f}')

    # Label notable agreements
    top5 = df_val.nlargest(3, 'oblig_ratio')
    bot5 = df_val.nsmallest(3, 'oblig_ratio')
    for _, row in pd.concat([top5, bot5]).iterrows():
        ax.annotate(str(row['agreement_name'])[:18],
                    (row['enf_count'], row['oblig_ratio']),
                    fontsize=6, xytext=(4, 2), textcoords='offset points')

    ax.set_xlabel('Hofmann et al. — Enforceable provision count (out of 52)')
    ax.set_ylabel('oblig_ratio (this study)')
    ax.set_title(f'Figure 5: External Validation against Hofmann et al. (2017)\n'
                 f'n={len(df_val)} agreements  |  Pearson r={r:.3f}  |  p={p:.4f}',
                 fontsize=10)
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig5_hofmann_validation.png'), bbox_inches='tight')
    plt.show()
    print("Figure 5 saved.")
else:
    print("Skipping Figure 5 — Hofmann data not loaded.")


---
## Part 16: SKELETON — Dyadic Variable Merge

**TODO:** Merge CEPII GeoDist and Bailey et al. UN voting data.

Sources:
- CEPII GeoDist: http://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=6
  Variables: `comlang_off` (common official language), `colony` (colonial history), `legal` (legal system)
  Merge key: `iso_o`, `iso_d` (ISO2 country codes) — match to df_meta `parties`
- Bailey, Strezhnev & Voeten (2017) UN voting ideal points:
  https://dataverse.harvard.edu/dataset.xhtml?persistentId=hdl:1902.1/12379
  Variables: `ideal_point` by country-year — compute cosine similarity for each dyad at signing year

For bilateral agreements: extract (party_a, party_b) from parties list.
For plurilateral: flag separately, use template dummy as primary control.


In [ ]:
# ── Dyadic variable merge — SKELETON ─────────────────────────────────────────
# Step 1: Extract country pairs from df_meta
df_dyad = df_meta[df_meta['n_parties'] == 2][['pta_id', 'parties', 'year']].copy()
df_dyad['party_a'] = df_dyad['parties'].apply(lambda p: p[0] if len(p) > 0 else None)
df_dyad['party_b'] = df_dyad['parties'].apply(lambda p: p[1] if len(p) > 1 else None)
df_dyad['year_int'] = pd.to_numeric(df_dyad['year'], errors='coerce').astype('Int64')

print(f"Bilateral agreements: {len(df_dyad)}")
print(df_dyad[['pta_id', 'party_a', 'party_b', 'year_int']].head(8).to_string(index=False))

# Step 2: Load CEPII GeoDist
# CEPII_PATH = 'dist_cepii.csv'
# if os.path.exists(CEPII_PATH):
#     df_cepii = pd.read_csv(CEPII_PATH)
#     df_cepii = df_cepii[['iso_o', 'iso_d', 'comlang_off', 'colony', 'legal']].copy()
#     # Merge both directions (A-B and B-A)
#     df_dyad = df_dyad.merge(df_cepii, left_on=['party_a','party_b'],
#                              right_on=['iso_o','iso_d'], how='left')
#     # Fill missing with reverse direction
#     # ... (add reverse merge logic here)
#     print(f"CEPII merged. common_language coverage: {df_dyad['comlang_off'].notna().sum()}/{len(df_dyad)}")

# Step 3: Load Bailey et al. UN voting ideal points
# BAILEY_PATH = 'IdealpointestimatesAll_Jul2024.tab'
# if os.path.exists(BAILEY_PATH):
#     df_bailey = pd.read_csv(BAILEY_PATH, sep='\t')
#     # Compute cosine similarity between ideal points for each dyad at signing year
#     # ... (add cosine similarity computation here)
#     print("Bailey et al. political alignment merged.")

print("\nSKELETON: add CEPII and Bailey et al. data to complete this section.")


---
## Part 17: SKELETON — Extension Regression

OLS regression of oblig_ratio on bilateral relationship features.

Spec: oblig_ratio ~ common_language + colonial_history + political_alignment
                  + common_legal_system
                  + d_us_tpa + d_eu_template + d_other_template
                  + d_eastern_european + d_central_asian

Three pre-registered robustness checks:
1. Symmetric subsample (asymmetry_type == 'bilateral_symmetric')
2. oblig_ratio_core as DV (Trade in Goods only)
3. Cluster interaction terms


In [ ]:
# ── Extension regression — SKELETON ──────────────────────────────────────────
# Requires dyadic variables from Part 16 to be merged first.

# import statsmodels.formula.api as smf

# df_analysis = df_agreement.copy()
# # TODO: merge dyadic vars from df_dyad into df_analysis on pta_id

# CONTROLS = ['d_us_tpa', 'd_eu_template', 'd_other_template',
#             'd_eastern_european', 'd_central_asian']
# DYADIC   = ['common_language', 'colonial_history', 'political_alignment',
#             'common_legal_system']

# # Main specification
# formula = 'oblig_ratio ~ ' + ' + '.join(DYADIC + CONTROLS)
# model   = smf.ols(formula, data=df_analysis.dropna(subset=DYADIC)).fit(cov_type='HC3')
# print(model.summary())

# # Robustness 1: symmetric subsample
# df_sym  = df_analysis[df_analysis['asymmetry_type'] == 'bilateral_symmetric']
# model_s = smf.ols(formula, data=df_sym.dropna(subset=DYADIC)).fit(cov_type='HC3')

# # Robustness 2: oblig_ratio_core as DV
# formula_core = formula.replace('oblig_ratio', 'oblig_ratio_core')
# model_c = smf.ols(formula_core, data=df_analysis.dropna(subset=DYADIC + ['oblig_ratio_core'])).fit(cov_type='HC3')

print("SKELETON: complete dyadic merge in Part 16 before running regression.")


---
## Part 18: Save Outputs

In [ ]:
# ── Save all outputs ──────────────────────────────────────────────────────────

# 1. Agreement-level analysis table
df_agreement.to_csv(os.path.join(OUTPUT_DIR, 'agreement_scores_v4.csv'), index=False)
print("Saved: agreement_scores_v4.csv")

# 2. Agreement metadata with asymmetry flags
df_meta.to_csv(os.path.join(OUTPUT_DIR, 'tota_metadata_v4.csv'), index=False)
print("Saved: tota_metadata_v4.csv")

# 3. Article-level corpus with all scores
save_cols = [
    'pta_id', 'agreement_name', 'year',
    'chapter_name', 'chapter_norm', 'chapter_index',
    'article_number', 'article_name', 'word_count',
    'hard_raw', 'hard_p1k', 'soft_raw', 'soft_p1k',
    'mixed_raw', 'mixed_p1k', 'enf_raw', 'enf_p1k',
    'soft_total', 'oblig_ratio',
]
save_cols = [c for c in save_cols if c in df_articles.columns]
df_articles[save_cols].to_csv(
    os.path.join(OUTPUT_DIR, 'corpus_articles_v4.csv'), index=False)
print("Saved: corpus_articles_v4.csv")

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("\n" + "=" * 55)
print("NEXT STEPS")
print("=" * 55)
print("  1. Run with max_ptas=None for full 423 English PTA corpus")
print("  2. Place Hofmann et al. CSV in working directory (Part 15)")
print("  3. Place CEPII GeoDist CSV in working directory (Part 16)")
print("  4. Place Bailey et al. TAB file in working directory (Part 16)")
print("  5. Complete dyadic merge and run regression (Part 17)")
